# Statistical analysis

In this notebook we will begin the analysis of the daya, dividing it into 4 sections: 

1. Formulate statistical questions
2. Hypothesis tests
3. Confidence intervals
14. ANOVA + post-hoc analysis

## Working days

- Does the average number of daily bikes rented differ between working and non-working days?
H0 = There is no difference in mean daily rentals between working and non-working days
H1 = There is no difference in mean daily rentals between working and non-working days
alpha = 0.05


In [2]:
import pandas as pd

data_days = pd.read_csv("../data/processed/data_days.csv", parse_dates=["dteday"])

In [3]:
#checking correct load
data_days.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [4]:
working_days = data_days[data_days.workingday == 1]
non_working_days = data_days[data_days.workingday == 0]

In [5]:
from scipy import stats

In [6]:
stats.ttest_ind(
    working_days["cnt"],
    non_working_days["cnt"],
    equal_var=False
)

TtestResult(statistic=np.float64(1.6013720686967898), pvalue=np.float64(0.11005737827017054), df=np.float64(413.93578279295593))

In [7]:
#testing for the difference in the two years

stats.ttest_ind(
    data_days.loc[data_days.yr == 0, "cnt"],
    data_days.loc[data_days.yr == 1, "cnt"],
    equal_var=False
)

TtestResult(statistic=np.float64(-18.57770740475173), pvalue=np.float64(1.0484385153559308e-62), df=np.float64(685.4970138571425))

## ANOVA 

In [8]:
stats.f_oneway(data_days[data_days["weathersit"] == 1]["cnt"], data_days[data_days["weathersit"] == 2]["cnt"], data_days[data_days["weathersit"] == 3]["cnt"])

F_onewayResult(statistic=np.float64(40.06604492024684), pvalue=np.float64(3.106317270053755e-17))

Seeing as we can reject the null hyopthesis, the interesting case now becomes which of the weathers is different from the rest. looking back at the exploration notebook, we knew that weathersit 1 would contain overall higher bike rentals, while wethersit 3 would have lowe rentals. We can confirm this by doing a ttsedt on both.

In [17]:
test_13 = stats.ttest_ind(
    data_days.loc[data_days["weathersit" ] == 3, "cnt"],
    data_days.loc[data_days["weathersit" ] == 1, "cnt"],
    equal_var=False
)

Seeing as p value is smaller than 0.05, we reject the null hypothesis, meaning a Clear, Few clouds, Partly cloudy, Partly cloudy day will have a different number of rentals to a day with Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds.

In [18]:
test_23 = stats.ttest_ind(
    data_days.loc[data_days["weathersit" ] == 3, "cnt"],
    data_days.loc[data_days["weathersit" ] == 2, "cnt"],
    equal_var=False
)

Seeing as p value is smaller than 0.05, we reject the null hypothesis, meaning a Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist day will have a different number of rentals to a Clear, Few clouds, Partly cloudy, Partly cloudy day.

In [19]:
test_12 = stats.ttest_ind(
    data_days.loc[data_days["weathersit"] == 2, "cnt"],
    data_days.loc[data_days["weathersit"] == 1, "cnt"],
    equal_var=False
)

Seeing as p value is smaller than 0.05, we reject the null hypothesis, meaning a Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist day will have a different number of rentals to a Clear, Few clouds, Partly cloudy, Partly cloudy


In [20]:
p_values = [test_12.pvalue, test_13.pvalue, test_23.pvalue]

In [25]:
from statsmodels.stats.multitest import multipletests
reject, p_adjusted, _, _ = multipletests(
    p_values, 
    alpha=0.05,
    method="holm"
)

In [26]:
print(p_adjusted)
print(reject)

[2.06697806e-08 2.64927550e-10 2.99614695e-08]
[ True  True  True]


After post-hoc analysis, we can tell that the difference in mean daily bike rentals is significantly different in all three weather types.

## Confidence Intervals 

In [28]:
weather_1 = data_days.loc[data_days["weathersit"] == 1, "cnt"]

mean = weather_1.mean()
sem = stats.sem(weather_1)
n = len(weather_1)

In [29]:
stats.t.interval(
    0.95,
    df=n-1,
    loc=mean,
    scale=sem
)

(np.float64(4705.139481558728), np.float64(5048.432872652935))

For Clear, Few clouds, Partly cloudy, Partly cloudy days, the confidence interval at 95%  for daily bike rentals is between 4705 and 5948. 

In [30]:
weather_2 = data_days.loc[data_days["weathersit"] == 2, "cnt"]
sem_2 = stats.sem(weather_2)
mean_2 = weather_2.mean()
n = len(weather_2)


In [33]:
stats.t.interval(
    0.95, 
    df=n-1, 
    loc=mean_2,
    scale=sem_2
)

(np.float64(3809.133668388691), np.float64(4262.5910279675845))

For days with Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist, the confidence interval at 95%  for daily bike rentals is between 3809 and 4263. 

In [35]:
weather_3 = data_days.loc[data_days["weathersit"] == 3, "cnt"]
mean = weather_3.mean()
n = len(weather_3)
sem = stats.sem(weather_3)

stats.t.interval(
    0.95, 
    df = n-1,
    loc = mean, 
    scale = sem
)

(np.float64(1238.7150018910093), np.float64(2367.856426680419))

For days with Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds, the confidence interval at 95%  for daily bike rentals is between 1238 and 2368. 

In [36]:
results = []

for weather in [1, 2, 3]:
    data = data_days.loc[data_days["weathersit"] == weather, "cnt"]
    
    mean = data.mean()
    sem = stats.sem(data)
    n = len(data)
    
    lower, upper = stats.t.interval(
        0.95,
        df=n-1,
        loc=mean,
        scale=sem
    )
    
    results.append({
        "Weather": weather,
        "Mean": mean,
        "CI Lower": lower,
        "CI Upper": upper
    })

ci_table = pd.DataFrame(results)
ci_table

,Weather,Mean,CI Lower,CI Upper
0,1,4876.786177,4705.139482,5048.432873
1,2,4035.862348,3809.133668,4262.591028
2,3,1803.285714,1238.715002,2367.856427


In the table we can observe how for each of the different weather situations, the mean is different, as well as how the 95% confidence intervals do not overlap, suggesting a difference in daily bike rentals between the weather situations.